# DeepFakeShield — Model Training v3 (Fixed)
Self-contained. No internet required. Works on CPU and GPU.
Author: vtangri | Repo: https://github.com/vtangri/DeepFakeShield

In [ ]:
# ── Cell 1: Verify environment ──────────────────────────────────────
import sys, os, json
import subprocess

print('Python:', sys.version)

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Using device:', DEVICE)

# Install missing packages quietly
pkgs = ['scikit-learn', 'matplotlib', 'seaborn', 'tqdm', 'opencv-python-headless']
for pkg in pkgs:
    r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', pkg],
                       capture_output=True, text=True)
    if r.returncode != 0:
        print(f'WARNING: failed to install {pkg}: {r.stderr[:200]}')
    else:
        print(f'OK: {pkg}')

print('Environment ready.')

In [ ]:
# ── Cell 2: Generate synthetic dataset ──────────────────────────────
import numpy as np
from pathlib import Path
import torch
import torchaudio
import cv2

DATA_DIR  = Path('/kaggle/temp/data')
MODEL_DIR = Path('/kaggle/working/models')
EVAL_DIR  = Path('/kaggle/working/evaluation')
for d in [DATA_DIR, MODEL_DIR, EVAL_DIR]: d.mkdir(parents=True, exist_ok=True)

SR = 16000
MAX_SEC = 3.0
IMG_SIZE = 112
N_FRAMES = 16
N_TRAIN, N_VAL, N_TEST = 80, 20, 20

def make_video(path, label):
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(str(path), fourcc, 10.0, (IMG_SIZE, IMG_SIZE))
    for i in range(N_FRAMES):
        if label == 0:
            f = np.zeros((IMG_SIZE, IMG_SIZE, 3), np.uint8)
            v = int(128 + 100*np.sin(2*np.pi*i/N_FRAMES))
            f[:,:,1] = np.clip(v, 0, 255)
            cv2.circle(f, (IMG_SIZE//2, IMG_SIZE//2), IMG_SIZE//3, (200,150,100), -1)
        else:
            f = np.random.randint(80, 180, (IMG_SIZE, IMG_SIZE, 3), np.uint8)
            cv2.line(f, (IMG_SIZE//2,0), (IMG_SIZE//2,IMG_SIZE), (0,0,255), 3)
        out.write(f)
    out.release()

def make_audio(path, label):
    n = int(SR * MAX_SEC)
    t = np.linspace(0, MAX_SEC, n)
    if label == 0:
        s = 0.5*np.sin(2*np.pi*220*t) + 0.3*np.sin(2*np.pi*440*t) + 0.05*np.random.randn(n)
    else:
        s = np.sin(2*np.pi*800*t)*np.sign(np.sin(2*np.pi*5*t)) + 0.3*np.random.randn(n)
        s = np.clip(s, -0.9, 0.9)
    s = s / (np.abs(s).max() + 1e-8)
    torchaudio.save(str(path), torch.tensor(s, dtype=torch.float32).unsqueeze(0), SR)

print('Generating dataset...')
for split, n in [('train', N_TRAIN), ('val', N_VAL), ('test', N_TEST)]:
    for label, name in [(0,'real'), (1,'fake')]:
        p = DATA_DIR/'video'/split/name; p.mkdir(parents=True, exist_ok=True)
        for i in range(n): make_video(p/f'{name}_{i:04d}.mp4', label)
    for label, name in [(0,'bonafide'), (1,'spoof')]:
        p = DATA_DIR/'audio'/split/name; p.mkdir(parents=True, exist_ok=True)
        for i in range(n): make_audio(p/f'{name}_{i:04d}.wav', label)
    print(f'  {split}: {n*2} video + {n*2} audio samples')

print('Dataset ready!')

In [ ]:
# ── Cell 3: Dataset classes ──────────────────────────────────────────
import torch
import numpy as np
import cv2
import torchaudio
from torch.utils.data import Dataset, DataLoader
from pathlib import Path

class VideoDS(Dataset):
    def __init__(self, root, split, size=224):
        self.size = size
        self.samples = []
        for lbl, name in [(0,'real'),(1,'fake')]:
            for p in (Path(root)/split/name).glob('*.mp4'):
                self.samples.append((str(p), lbl))
        print(f'VideoDS [{split}]: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        cap = cv2.VideoCapture(path)
        total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT)) or 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, total // 2)
        ret, frame = cap.read()
        cap.release()
        if not ret:
            frame = np.zeros((self.size, self.size, 3), np.uint8)
        frame = cv2.resize(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), (self.size, self.size))
        t = torch.from_numpy(frame).permute(2,0,1).float() / 255.0
        mean = torch.tensor([0.485,0.456,0.406]).view(3,1,1)
        std  = torch.tensor([0.229,0.224,0.225]).view(3,1,1)
        return (t - mean) / std, torch.tensor(lbl, dtype=torch.float32)

class AudioDS(Dataset):
    def __init__(self, root, split, sr=16000, max_sec=3.0):
        self.sr = sr
        self.max_n = int(sr * max_sec)
        self.samples = []
        for lbl, name in [(0,'bonafide'),(1,'spoof')]:
            for p in (Path(root)/split/name).glob('*.wav'):
                self.samples.append((str(p), lbl))
        print(f'AudioDS [{split}]: {len(self.samples)} samples')

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, lbl = self.samples[idx]
        try:
            w, sr = torchaudio.load(path)
            if sr != self.sr:
                w = torchaudio.transforms.Resample(sr, self.sr)(w)
            if w.shape[0] > 1: w = w.mean(0, keepdim=True)
            if w.shape[1] > self.max_n: w = w[:, :self.max_n]
            else: w = torch.nn.functional.pad(w, (0, self.max_n - w.shape[1]))
        except Exception as e:
            print(f'WARN audio load: {e}')
            w = torch.zeros(1, self.max_n)
        return w.squeeze(0), torch.tensor(lbl, dtype=torch.float32)

def make_loaders(DS, root, bs, nw=0):
    # num_workers=0 avoids fork issues in Kaggle API kernels
    # pin_memory only when CUDA available
    pm = torch.cuda.is_available()
    tr = DataLoader(DS(root,'train'), bs, shuffle=True,  num_workers=nw, pin_memory=pm)
    vl = DataLoader(DS(root,'val'),   bs, shuffle=False, num_workers=nw)
    te = DataLoader(DS(root,'test'),  bs, shuffle=False, num_workers=nw)
    return tr, vl, te

print('Dataset classes defined.')

In [ ]:
# ── Cell 4: Model architectures ──────────────────────────────────────
import torch.nn as nn
import torchaudio
from torchvision.models import vit_b_16, ViT_B_16_Weights

def build_video_model():
    # Pretrained weights require internet (phone-verified Kaggle account).
    # Fall back to random init so the run still completes offline.
    try:
        m = vit_b_16(weights=ViT_B_16_Weights.DEFAULT)
        print('Video backbone: ImageNet-pretrained ViT-B/16')
    except Exception as e:
        print(f'WARNING: could not fetch pretrained ViT weights ({type(e).__name__}: {e}).')
        print('Falling back to randomly-initialised ViT-B/16.')
        m = vit_b_16(weights=None)
    m.heads = nn.Sequential(
        nn.Linear(768, 256), nn.ReLU(), nn.Dropout(0.3),
        nn.Linear(256, 1),   nn.Sigmoid()
    )
    return m

class AudioModel(nn.Module):
    def __init__(self, sr=16000):
        super().__init__()
        self.mel = torchaudio.transforms.MelSpectrogram(
            sample_rate=sr, n_fft=1024, hop_length=256, n_mels=80)
        self.features = nn.Sequential(
            nn.Conv2d(1,  32, 3, padding=1), nn.BatchNorm2d(32),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64),  nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64,128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(128,256,3, padding=1), nn.BatchNorm2d(256), nn.ReLU(),
            nn.AdaptiveAvgPool2d((4,4)),
        )
        self.head = nn.Sequential(
            nn.Flatten(), nn.Linear(256*16, 256), nn.ReLU(), nn.Dropout(0.5),
            nn.Linear(256,1), nn.Sigmoid()
        )
    def forward(self, x):
        m = self.mel(x).unsqueeze(1)
        m = (m - m.mean()) / (m.std() + 1e-8)
        return self.head(self.features(m))

print('Models defined.')

In [ ]:
# ── Cell 5: Training helpers ─────────────────────────────────────────
import torch
from tqdm import tqdm   # plain tqdm, not tqdm.notebook — works everywhere

def run_epoch(model, loader, crit, opt, device, train=True):
    model.train(train)
    loss_sum = correct = total = 0
    ctx = torch.enable_grad() if train else torch.no_grad()
    preds_all, labels_all = [], []
    with ctx:
        for x, y in tqdm(loader, desc='train' if train else 'eval', leave=False):
            x, y = x.to(device), y.to(device).unsqueeze(1)
            out = model(x)
            loss = crit(out, y)
            if train:
                opt.zero_grad(); loss.backward(); opt.step()
            loss_sum += loss.item()
            correct  += ((out > 0.5).float() == y).sum().item()
            total    += y.size(0)
            preds_all.extend(out.detach().cpu().numpy().flatten())
            labels_all.extend(y.detach().cpu().numpy().flatten())
    return loss_sum/len(loader), correct/total, preds_all, labels_all

def train_model(model, tr, vl, te, device, epochs, save_path, name):
    import torch.optim as optim
    crit = torch.nn.BCELoss()
    opt  = optim.Adam(model.parameters(), lr=1e-3)
    sched= optim.lr_scheduler.ReduceLROnPlateau(opt, patience=3)
    best_acc, results = 0, []
    print(f'\n=== Training {name} for {epochs} epochs on {device} ===')
    for ep in range(1, epochs+1):
        tl, ta, _, _ = run_epoch(model, tr, crit, opt, device, train=True)
        vl2, va, _, _= run_epoch(model, vl, crit, opt, device, train=False)
        sched.step(vl2)
        results.append({'epoch':ep,'train_loss':tl,'train_acc':ta,'val_loss':vl2,'val_acc':va})
        print(f'  Ep {ep:02d}/{epochs} | train loss={tl:.4f} acc={ta:.4f} | val loss={vl2:.4f} acc={va:.4f}')
        if va > best_acc:
            best_acc = va
            torch.save(model.state_dict(), save_path)
            print(f'    -> Saved best ({save_path.name}, val_acc={va:.4f})')
    # Test
    model.load_state_dict(torch.load(save_path, map_location=device))
    _, test_acc, test_preds, test_labels = run_epoch(model, te, crit, opt, device, train=False)
    print(f'  Test accuracy: {test_acc:.4f}')
    return test_acc, test_preds, test_labels, results

print('Training helpers defined.')

In [ ]:
# ── Cell 6: Train Audio Spoof Model ──────────────────────────────────
device = torch.device(DEVICE)

audio_tr, audio_vl, audio_te = make_loaders(AudioDS, DATA_DIR/'audio', bs=16)

audio_model = AudioModel().to(device)
audio_path  = MODEL_DIR / 'audio_spoof_final.pt'

audio_test_acc, audio_preds, audio_labels, audio_history = train_model(
    audio_model, audio_tr, audio_vl, audio_te,
    device, epochs=15, save_path=audio_path, name='AudioSpoofCNN'
)
print(f'Audio model saved to {audio_path}')

In [ ]:
# ── Cell 7: Train Video Forensics Model (ViT-B/16) ───────────────────
video_tr, video_vl, video_te = make_loaders(VideoDS, DATA_DIR/'video', bs=8)

video_model = build_video_model().to(device)
video_path  = MODEL_DIR / 'video_forensics_final.pt'

video_test_acc, video_preds, video_labels, video_history = train_model(
    video_model, video_tr, video_vl, video_te,
    device, epochs=8, save_path=video_path, name='VideoViT-B16'
)
print(f'Video model saved to {video_path}')

In [ ]:
# ── Cell 8: Plot & save evaluation charts ────────────────────────────
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — works in all environments
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import roc_curve, auc, confusion_matrix, classification_report

def save_eval_charts(name, preds, labels, out_dir):
    labels_bin = [int(l) for l in labels]
    preds_bin  = [1 if p > 0.5 else 0 for p in preds]
    # ROC
    fpr, tpr, _ = roc_curve(labels_bin, preds)
    roc_auc = auc(fpr, tpr)
    fig, ax = plt.subplots(figsize=(6,5))
    ax.plot(fpr, tpr, lw=2, label=f'ROC AUC = {roc_auc:.4f}')
    ax.plot([0,1],[0,1],'k--')
    ax.set(xlabel='FPR', ylabel='TPR', title=f'{name} ROC Curve')
    ax.legend(); fig.tight_layout()
    fig.savefig(out_dir/f'{name}_roc.png', dpi=100); plt.close(fig)
    # Confusion matrix
    cm = confusion_matrix(labels_bin, preds_bin)
    fig, ax = plt.subplots(figsize=(5,4))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Real','Fake'], yticklabels=['Real','Fake'], ax=ax)
    ax.set_title(f'{name} Confusion Matrix'); fig.tight_layout()
    fig.savefig(out_dir/f'{name}_cm.png', dpi=100); plt.close(fig)
    # Classification report
    report = classification_report(labels_bin, preds_bin, target_names=['Real','Fake'])
    (out_dir/f'{name}_report.txt').write_text(report)
    # Metrics JSON
    with open(out_dir/f'{name}_metrics.json','w') as f:
        json.dump({'roc_auc': roc_auc, 'test_accuracy': float(sum(p==l for p,l in zip(preds_bin,labels_bin)))/len(labels_bin)}, f, indent=2)
    print(f'{name}: AUC={roc_auc:.4f}')
    print(report)

save_eval_charts('audio', audio_preds, audio_labels, EVAL_DIR)
save_eval_charts('video', video_preds, video_labels, EVAL_DIR)
print('All evaluation charts saved.')

In [ ]:
# ── Cell 9: Summary of all output files ──────────────────────────────
import os
print('=== Trained Models ===')
for f in sorted(MODEL_DIR.glob('*.pt')):
    print(f'  {f.name:45s}  {f.stat().st_size/1024/1024:.1f} MB')

print('\n=== Evaluation Outputs ===')
for f in sorted(EVAL_DIR.iterdir()):
    print(f'  {f.name}')

print('\n=== Final Accuracies ===')
print(f'  Audio test accuracy : {audio_test_acc:.4f}')
print(f'  Video test accuracy : {video_test_acc:.4f}')
print('\nTraining complete!')